# Pothole data sandbox

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()

# Create a view to the pothole data
RAW_311 = 'data/raw/311_pothole.parquet'
con.sql(f"""
    CREATE OR REPLACE VIEW raw_311 AS
    SELECT * FROM '{RAW_311}'
""")

# Print version
con.sql("SELECT version()").df()

,"""version""()"
0,v1.5.3


In [2]:
# Describe the table
con.sql("DESCRIBE raw_311").show(max_rows=40)

┌────────────────────────────────┬──────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name           │                 column_type                  │  null   │   key   │ default │  extra  │
│            varchar             │                   varchar                    │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────────┼──────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ unique_key                     │ VARCHAR                                      │ YES     │ NULL    │ NULL    │ NULL    │
│ created_date                   │ TIMESTAMP                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ closed_date                    │ TIMESTAMP                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ agency                         │ VARCHAR                                      │ YES     │ NULL    │ NULL    │ NULL    │
│ agency_name           

In [3]:
# Basic select everything
con.sql("SELECT COUNT(*) FROM raw_311").df()

,count_star()
0,75058


In [4]:
con.sql("""
        SELECT descriptor, COUNT(*) AS n_rows
        FROM raw_311
        GROUP BY descriptor
        ORDER BY n_rows DESC
    """)

┌────────────┬────────┐
│ descriptor │ n_rows │
│  varchar   │ int64  │
├────────────┼────────┤
│ Pothole    │  75058 │
└────────────┴────────┘

In [5]:
con.sql("""
    SELECT borough, COUNT(*) AS n_rows
    FROM raw_311
    GROUP BY borough
    ORDER BY n_rows DESC
    """)

┌───────────────┬────────┐
│    borough    │ n_rows │
│    varchar    │ int64  │
├───────────────┼────────┤
│ QUEENS        │  29376 │
│ BROOKLYN      │  17626 │
│ MANHATTAN     │  10316 │
│ BRONX         │   8795 │
│ STATEN ISLAND │   8723 │
│ Unspecified   │    222 │
└───────────────┴────────┘

In [6]:
con.sql("""
    SELECT status, COUNT(*) AS n_rows
    FROM raw_311
    GROUP BY status
    ORDER BY n_rows DESC
    """)

┌─────────────┬────────┐
│   status    │ n_rows │
│   varchar   │ int64  │
├─────────────┼────────┤
│ Closed      │  72256 │
│ Pending     │   2799 │
│ Unspecified │      3 │
└─────────────┴────────┘

In [7]:
con.sql("""
    SELECT MIN(created_date), MAX(created_date),
        COUNT(*) FILTER (WHERE created_date IS NULL) AS n_null
    FROM raw_311
""")

┌─────────────────────┬─────────────────────┬────────┐
│  min(created_date)  │  max(created_date)  │ n_null │
│      timestamp      │      timestamp      │ int64  │
├─────────────────────┼─────────────────────┼────────┤
│ 2023-01-01 09:15:59 │ 2024-12-31 23:24:25 │      0 │
└─────────────────────┴─────────────────────┴────────┘

In [8]:
con.sql("""
    SELECT COUNT(*) FILTER (WHERE longitude IS NULL
        OR latitude IS NULL) AS n_no_coord
    FROM raw_311
""")

┌────────────┐
│ n_no_coord │
│   int64    │
├────────────┤
│      34435 │
└────────────┘

In [9]:
con.sql("""
    SELECT COUNT(*) AS dupe_count
    FROM (SELECT latitude, longitude, created_date
        FROM raw_311 
        GROUP BY 1,2,3 HAVING COUNT(*) > 1)
""")

┌────────────┐
│ dupe_count │
│   int64    │
├────────────┤
│        822 │
└────────────┘

In [10]:
con.sql("""
    SELECT borough, COUNT(*) FILTER (WHERE latitude IS NULL) AS n_missing, COUNT(*) AS n_total, COUNT(*) FILTER (WHERE latitude IS NULL) * 100.0 / COUNT(*) AS pct_missing FROM raw_311 GROUP BY borough ORDER BY pct_missing DESC
""").df()

,borough,n_missing,n_total,pct_missing
0,Unspecified,194,222,87.387387
1,QUEENS,15118,29376,51.463780
2,STATEN ISLAND,3976,8723,45.580649
3,BROOKLYN,7418,17626,42.085555
4,BRONX,3673,8795,41.762365
5,MANHATTAN,4056,10316,39.317565


In [11]:
con.sql("""
    SELECT DATE_TRUNC('month', created_date) AS month, COUNT(*) FILTER (WHERE latitude IS NULL) * 100.0 / COUNT(*) AS pct_missing FROM raw_311 GROUP BY 1 ORDER BY 1
""").df()

,month,pct_missing
0,2023-01-01,38.202934
1,2023-02-01,37.799658
2,2023-03-01,40.492505
3,2023-04-01,57.911268
4,2023-05-01,56.390449
5,2023-06-01,41.743255
6,2023-07-01,49.669967
7,2023-08-01,49.614862
8,2023-09-01,47.315258
9,2023-10-01,48.643503


In [12]:
con.sql("""
    SELECT open_data_channel_type, COUNT(*) FILTER (WHERE latitude IS NULL) * 100.0 / COUNT(*) AS pct_missing, COUNT(*) AS n FROM raw_311 GROUP BY 1 ORDER BY n DESC
""").df()

,open_data_channel_type,pct_missing,n
0,UNKNOWN,45.877854,75058


In [13]:
con.sql("""
    SELECT 
    COUNT(*) AS n_missing_coords,
    COUNT(*) FILTER (WHERE incident_zip IS NOT NULL AND incident_zip != '') AS n_have_zip,
    COUNT(*) FILTER (WHERE incident_address IS NOT NULL AND incident_address != '') AS n_have_address,
    COUNT(*) FILTER (WHERE community_board IS NOT NULL AND community_board != '') AS n_comm_board,
    COUNT(*) FILTER (WHERE police_precinct IS NOT NULL AND police_precinct != '') AS n_precinct
FROM raw_311
WHERE latitude IS NULL OR longitude IS NULL
""").df()

,n_missing_coords,n_have_zip,n_have_address,n_comm_board,n_precinct
0,34435,32993,34187,34435,34435


In [14]:
# Check for orthogonality between address_type values - BLOCKFACE, INTERSECTION, ADDRESS are coded differently
con.sql("""
    SELECT
    COUNT(*) FILTER (WHERE latitude IS NULL AND incident_address IS NOT NULL) * 100 / COUNT(*) AS pct_lat_no_address,
    COUNT(*) FILTER (WHERE latitude IS NOT NULL AND incident_address IS NULL) * 100 / COUNT(*) AS pct_no_lat_address,
    COUNT(*) FILTER (WHERE latitude IS NULL AND incident_address IS NULL) * 100 / COUNT(*) AS pct_no_lat_no_address,
    from RAW_311
""").df()

,pct_lat_no_address,pct_no_lat_address,pct_no_lat_no_address
0,45.547443,36.934104,0.330411


In [15]:
pd.set_option('display.max_rows', None)

con.sql("""
    SELECT community_board, COUNT(*) as n_rows
    FROM raw_311
    GROUP BY community_board
    ORDER BY community_board
""").df()

,community_board,n_rows
0,0 Unspecified,222
1,01 BRONX,611
2,01 BROOKLYN,1825
3,01 MANHATTAN,963
4,01 QUEENS,2204
5,01 STATEN ISLAND,2423
6,02 BRONX,306
7,02 BROOKLYN,1101
8,02 MANHATTAN,755
9,02 QUEENS,1856


In [16]:
con.sql("""
    SELECT 
        COUNT(*) FILTER (WHERE incident_zip IS NULL) AS n_null_zip,
        COUNT(*) FILTER (WHERE incident_zip = '') AS n_empty_zip,
        COUNT(*) FILTER (WHERE address_type IS NULL) AS n_null_addr_type,
        COUNT(*) FILTER (WHERE address_type = '') AS n_empty_addr_type
    FROM raw_311
""").df()

,n_null_zip,n_empty_zip,n_null_addr_type,n_empty_addr_type
0,1442,0,0,0


In [17]:
con.sql("""
    SELECT 
        COUNT(*) FILTER (WHERE community_board LIKE '0 %' AND latitude IS NOT NULL) AS coord_no_board,
    FROM raw_311
""").df()

,coord_no_board
0,28


In [35]:
#pd.set_option('display.max_rows', None)
pd.reset_option('display.max_rows')
tsg = """
WITH source AS (
    -- Pull data from raw parquet file, select columns
    SELECT 
        unique_key,
        created_date,
        closed_date,
        status,
        descriptor,
        borough,
        community_board,
        incident_zip,
        incident_address,
        address_type,
        latitude,
        longitude
    FROM 'data/raw/311_pothole.parquet'
), complaints_projected AS (
    -- fix any types, rename/select columns if needed
    SELECT
        unique_key,
        created_date,
        closed_date,
        status,
        borough,            -- already uppercase in source
        community_board,    -- "NN BOROUGH" format, parse later
        incident_zip,
        address_type,       -- already uppercase in source
        latitude,
        longitude
    FROM source
), complaints_filtered AS (
    -- Enforce time window, key existence, some geospatial data
    SELECT *
    FROM complaints_projected
    WHERE
        (created_date >= CAST('2023-01-01' AS TIMESTAMP) AND created_date < CAST('2025-01-01' AS TIMESTAMP))
        AND (unique_key IS NOT NULL)
        AND (
            (longitude IS NOT NULL AND latitude IS NOT NULL) 
            OR community_board NOT LIKE '0 %'
            )
), complaints_enriched AS (
    -- Transform community board string to BoroCD identifiers, flags for interesting locales
    SELECT *,
        -- BoroCD identifiers
        CASE 
            WHEN community_board LIKE '%Unspecified%'
                THEN NULL
            ELSE CAST(
                CASE
                    WHEN community_board LIKE '% MANHATTAN' THEN '1'
                    WHEN community_board LIKE '% BRONX' THEN '2'
                    WHEN community_board LIKE '% BROOKLYN' THEN '3'
                    WHEN community_board LIKE '% QUEENS' THEN '4'
                    WHEN community_board LIKE '% STATEN ISLAND' THEN '5'
                END || SPLIT_PART(community_board, ' ', 1)
                AS INTEGER 
            )
        END AS borocd,
        -- Joint Interest Area flag (JIA; parks etc)
        CASE
            WHEN community_board LIKE '%Unspecified%'
                THEN NULL
            ELSE CAST(
                SPLIT_PART(community_board, ' ', 1) 
                AS INTEGER
                ) >= 20
        END AS is_jia,
        -- Report month
        DATE_TRUNC('month', created_date) AS report_month,
        -- Time to close
        DATE_DIFF('day', created_date, closed_date) AS days_to_close
    FROM complaints_filtered
), complaints_ranked AS (
    SELECT *,
    CASE
        WHEN latitude IS NOT NULL
            THEN ROW_NUMBER() OVER (
                PARTITION BY latitude, longitude, DATE(created_date)
                ORDER BY created_date ASC
            )
        ELSE 1 -- no-coord rows cant be determined if duplicate; keep all
    END AS row_num
    FROM complaints_enriched
), complaints_deduped AS (
    SELECT *
    FROM complaints_ranked
    WHERE row_num = 1
) SELECT * EXCLUDE (row_num) FROM complaints_deduped ORDER BY unique_key
"""
con.sql(tsg).df()

,unique_key,created_date,closed_date,status,borough,community_board,incident_zip,address_type,latitude,longitude,borocd,is_jia,report_month,days_to_close
0,56412791,2023-01-01 19:28:23,2023-01-03 12:15:00,Closed,QUEENS,08 QUEENS,11432,BLOCKFACE,NaN,NaN,408,False,2023-01-01,2
1,56413018,2023-01-01 21:48:16,2023-01-03 09:50:00,Closed,MANHATTAN,05 MANHATTAN,10016,BLOCKFACE,NaN,NaN,105,False,2023-01-01,2
2,56413020,2023-01-01 21:40:26,2023-01-03 08:25:00,Closed,QUEENS,10 QUEENS,11414,INTERSECTION,40.672792,-73.856915,410,False,2023-01-01,2
3,56413022,2023-01-01 10:16:01,2023-01-03 10:52:00,Closed,STATEN ISLAND,03 STATEN ISLAND,10306,INTERSECTION,40.546812,-74.159000,503,False,2023-01-01,2
4,56413748,2023-01-01 14:47:53,2023-01-03 09:25:00,Closed,BROOKLYN,10 BROOKLYN,11228,BLOCKFACE,NaN,NaN,310,False,2023-01-01,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72635,63621430,2024-12-31 06:52:59,2025-01-02 13:05:00,Closed,BRONX,Unspecified BRONX,NaN,BLOCKFACE,NaN,NaN,<NA>,<NA>,2024-12-01,2
72636,63621463,2024-12-31 06:43:38,2025-01-02 13:14:00,Closed,BRONX,12 BRONX,10470,INTERSECTION,40.896591,-73.870069,212,False,2024-12-01,2
72637,63621464,2024-12-31 06:40:33,2025-01-02 10:10:00,Closed,BRONX,12 BRONX,10466,INTERSECTION,40.898665,-73.848720,212,False,2024-12-01,2
72638,63621550,2024-12-31 07:09:05,2025-01-02 09:25:00,Closed,BRONX,09 BRONX,10472,ADDRESS,40.831829,-73.866488,209,False,2024-12-01,2


In [36]:
pd.reset_option('display.max_rows')

# Try running my SQL script
msg = open('sql/01_clean_311.sql').read()

con.sql(msg).df()

,unique_key,created_date,closed_date,status,borough,community_board,incident_zip,address_type,latitude,longitude,borocd,is_jia,report_month,days_to_close
0,56412791,2023-01-01 19:28:23,2023-01-03 12:15:00,Closed,QUEENS,08 QUEENS,11432,BLOCKFACE,NaN,NaN,408,False,2023-01-01,2
1,56413018,2023-01-01 21:48:16,2023-01-03 09:50:00,Closed,MANHATTAN,05 MANHATTAN,10016,BLOCKFACE,NaN,NaN,105,False,2023-01-01,2
2,56413020,2023-01-01 21:40:26,2023-01-03 08:25:00,Closed,QUEENS,10 QUEENS,11414,INTERSECTION,40.672792,-73.856915,410,False,2023-01-01,2
3,56413022,2023-01-01 10:16:01,2023-01-03 10:52:00,Closed,STATEN ISLAND,03 STATEN ISLAND,10306,INTERSECTION,40.546812,-74.159000,503,False,2023-01-01,2
4,56413748,2023-01-01 14:47:53,2023-01-03 09:25:00,Closed,BROOKLYN,10 BROOKLYN,11228,BLOCKFACE,NaN,NaN,310,False,2023-01-01,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72635,63621430,2024-12-31 06:52:59,2025-01-02 13:05:00,Closed,BRONX,Unspecified BRONX,NaN,BLOCKFACE,NaN,NaN,<NA>,<NA>,2024-12-01,2
72636,63621463,2024-12-31 06:43:38,2025-01-02 13:14:00,Closed,BRONX,12 BRONX,10470,INTERSECTION,40.896591,-73.870069,212,False,2024-12-01,2
72637,63621464,2024-12-31 06:40:33,2025-01-02 10:10:00,Closed,BRONX,12 BRONX,10466,INTERSECTION,40.898665,-73.848720,212,False,2024-12-01,2
72638,63621550,2024-12-31 07:09:05,2025-01-02 09:25:00,Closed,BRONX,09 BRONX,10472,ADDRESS,40.831829,-73.866488,209,False,2024-12-01,2


In [24]:
pd.reset_option('display.max_rows')

con.sql(f"""
    SELECT *,
    ROW_NUMBER() OVER (
        PARTITION BY latitude, longitude, DATE(created_date)
        ORDER BY created_date ASC
        ) AS row_num
    FROM raw_311
    ORDER BY unique_key ASC
""").df()

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,incident_zip,incident_address,street_name,...,intersection_street_1,intersection_street_2,council_district,x_coordinate_state_plane,y_coordinate_state_plane,latitude,longitude,location,bbl,row_num
0,56412791,2023-01-01 19:28:23,2023-01-03 12:15:00,DOT,Department of Transportation,Street Condition,Pothole,11432,GRAND CENTRL PARKWAY,GRAND CENTRL PARKWAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,23
1,56413018,2023-01-01 21:48:16,2023-01-03 09:50:00,DOT,Department of Transportation,Street Condition,Pothole,10016,MADISON AVENUE,MADISON AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,29
2,56413020,2023-01-01 21:40:26,2023-01-03 08:25:00,DOT,Department of Transportation,Street Condition,Pothole,11414,NaN,NaN,...,79 STREET,SOUTH CONDUIT AVENUE,32,1023941.0,184425.0,40.672792,-73.856915,"{'type': 'Point', 'coordinates': [-73.85691464...",NaN,1
3,56413022,2023-01-01 10:16:01,2023-01-03 10:52:00,DOT,Department of Transportation,Street Condition,Pothole,10306,AMBOY ROAD,AMBOY ROAD,...,ARMSTRONG AVENUE,OLD AMBOY ROAD,51,940061.0,138535.0,40.546812,-74.159000,"{'type': 'Point', 'coordinates': [-74.15900025...",NaN,1
4,56413748,2023-01-01 14:47:53,2023-01-03 09:25:00,DOT,Department of Transportation,Street Condition,Pothole,11228,12 AVENUE,12 AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75053,63621430,2024-12-31 06:52:59,2025-01-02 13:05:00,DOT,Department of Transportation,Street Condition,Pothole,NaN,BOSTON ROAD,BOSTON ROAD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,5
75054,63621463,2024-12-31 06:43:38,2025-01-02 13:14:00,DOT,Department of Transportation,Street Condition,Pothole,10470,NaN,NaN,...,EAST 233 STREET,EAST 234 STREET,11,1020171.0,265957.0,40.896591,-73.870069,"{'type': 'Point', 'coordinates': [-73.87006944...",NaN,1
75055,63621464,2024-12-31 06:40:33,2025-01-02 10:10:00,DOT,Department of Transportation,Street Condition,Pothole,10466,NaN,NaN,...,ELY AVENUE,NEREID AVENUE,12,1026072.0,266722.0,40.898665,-73.848720,"{'type': 'Point', 'coordinates': [-73.84872007...",NaN,1
75056,63621550,2024-12-31 07:09:05,2025-01-02 09:25:00,DOT,Department of Transportation,Street Condition,Pothole,10472,1798 WESTCHESTER AVENUE,WESTCHESTER AVENUE,...,NaN,NaN,18,1021197.0,242363.0,40.831829,-73.866488,"{'type': 'Point', 'coordinates': [-73.86648835...",2037630038,1


In [25]:
pd.reset_option('display.max_rows')

con.sql(f"""
    SELECT 
    SUM(n_in_group - 1) AS rows_to_drop
FROM (
    SELECT latitude, longitude, DATE(created_date), COUNT(*) AS n_in_group
    FROM raw_311
    WHERE latitude IS NOT NULL  -- can't dedupe rows without coordinates
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
)
""").df()

,rows_to_drop
0,2224.0
